<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/notebook1_patient_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Patient Analysis (PPMI)

**Objective:** fully analyze `Participant_Status.csv` and `Pacientes.csv` to understand the patient population, validate the common identifier that will be used in the rest of the project, and generate `participant_index.csv` as a light output.

No database or interface is built. Just exploratory analysis + index generation.

In [30]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# ============================================================
# Project paths
# ============================================================
BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"

PARTICIPANT_STATUS = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/primera entrega/Conteo de los pacientes/Participant_Status.csv"
PACIENTES_CSV      = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/primera entrega//Conteo de los pacientes/Pacientes.csv"

RESULTADOS_DIR = BASE_DIR + "/Resultados"

import os
os.makedirs(RESULTADOS_DIR, exist_ok=True)
print("Results will be saved in:", RESULTADOS_DIR)

Results will be saved in: /content/drive/MyDrive/Investigación_Parkinson/Resultados


In [32]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


## 1. Loading Participant_Status.csv (main reference file)

In [33]:
def cargar_csv(path, nombre):
    """Loads a CSV and displays basic diagnostic information."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"'{nombre}' not found at: {path}")
    df = pd.read_csv(path, low_memory=False)
    print(f"--- {nombre} ---")
    print("Rows:", df.shape[0], "| Columns:", df.shape[1])
    print("Memory usage: {:.2f} MB".format(df.memory_usage(deep=True).sum() / 1024**2))
    return df

df_status = cargar_csv(PARTICIPANT_STATUS, "Participant_Status.csv")
df_status.head()

--- Participant_Status.csv ---
Rows: 8619 | Columns: 33
Memory usage: 4.43 MB


,PATNO,COHORT,COHORT_DEFINITION,ENROLL_DATE,ENROLL_STATUS,STATUS_DATE,SCREENEDAM,ENROLL_AGE,INEXPAGE,AV133STDY,TAUSTDY,GAITSTDY,PISTDY,SV2ASTDY,NXTAUSTDY,DPPDSTDY,DPPROSTDY,FD4STDY,DPASTDY,DPDPASTDY,GAITLEAPSTDY,DATELIG,PPMI_ONLINE_ENROLL,ENRLPINK1,ENRLPRKN,ENRLSRDC,ENRLNORM,ENRLOTHGV,ENRLHPSM,ENRLRBD,ENRLLRRK2,ENRLSNCA,ENRLGBA
0,3000,2,Healthy Control,02/2011,Withdrew,10/2024,NaN,69.1,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
1,3001,1,Parkinson's Disease,03/2011,Withdrew,01/2026,NaN,65.1,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
2,3002,1,Parkinson's Disease,03/2011,Withdrew,10/2024,NaN,67.6,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
3,3003,1,Parkinson's Disease,04/2011,Enrolled,01/2022,NaN,56.7,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
4,3004,2,Healthy Control,04/2011,Enrolled,01/2022,NaN,59.4,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0


In [34]:
print("Available columns in Participant_Status.csv:\n")
for c in df_status.columns:
    print(" -", c)

Available columns in Participant_Status.csv:

 - PATNO
 - COHORT
 - COHORT_DEFINITION
 - ENROLL_DATE
 - ENROLL_STATUS
 - STATUS_DATE
 - SCREENEDAM
 - ENROLL_AGE
 - INEXPAGE
 - AV133STDY
 - TAUSTDY
 - GAITSTDY
 - PISTDY
 - SV2ASTDY
 - NXTAUSTDY
 - DPPDSTDY
 - DPPROSTDY
 - FD4STDY
 - DPASTDY
 - DPDPASTDY
 - GAITLEAPSTDY
 - DATELIG
 - PPMI_ONLINE_ENROLL
 - ENRLPINK1
 - ENRLPRKN
 - ENRLSRDC
 - ENRLNORM
 - ENRLOTHGV
 - ENRLHPSM
 - ENRLRBD
 - ENRLLRRK2
 - ENRLSNCA
 - ENRLGBA


In [35]:
# Data types and nulls per column
resumen_status = pd.DataFrame({
    "dtype": df_status.dtypes.astype(str),
    "n_nulos": df_status.isna().sum(),
    "pct_nulos": (df_status.isna().mean() * 100).round(2),
    "n_unicos": df_status.nunique()
}).sort_values("pct_nulos", ascending=False)

resumen_status

,dtype,n_nulos,pct_nulos,n_unicos
DATELIG,float64,5407,62.73,2
ENROLL_AGE,float64,3771,43.75,475
ENROLL_DATE,object,3760,43.62,179
INEXPAGE,object,2129,24.70,7
SCREENEDAM,float64,2129,24.70,3
ENRLOTHGV,float64,853,9.90,2
SV2ASTDY,float64,853,9.90,2
GAITSTDY,float64,853,9.90,2
PISTDY,float64,853,9.90,2
DPPDSTDY,float64,853,9.90,2


### How many patients exist and how many are `Enrolled`?

In [36]:
assert "PATNO" in df_status.columns, "PATNO column not found in Participant_Status.csv"

n_total_status = df_status["PATNO"].nunique()
print("Unique patients (PATNO) in Participant_Status:", n_total_status)

if "ENROLL_STATUS" in df_status.columns:
    print("\nENROLL_STATUS distribution:")
    print(df_status["ENROLL_STATUS"].value_counts(dropna=False))

    pacientes_activos = df_status[df_status["ENROLL_STATUS"] == "Enrolled"].copy()
    print("\nPatients with ENROLL_STATUS = 'Enrolled':", pacientes_activos["PATNO"].nunique())
elif "ENROLL_STATUS" not in df_status.columns:
    print("WARNING: ENROLL_STATUS column does not exist. Check the exact name in the CSV.")
    pacientes_activos = pd.DataFrame(columns=df_status.columns)

Unique patients (PATNO) in Participant_Status: 8619

ENROLL_STATUS distribution:
ENROLL_STATUS
Enrolled             3999
Screen failed        3040
Withdrew              592
Excluded              271
Screened              141
Pending               140
Declined              114
Complete              101
Withdraw Deceased      84
Baseline Withdraw      50
Screen Scheduled       49
Baseline               36
Lost to follow-up       1
NaN                     1
Name: count, dtype: int64

Patients with ENROLL_STATUS = 'Enrolled': 3999


### What cohorts exist and how many patients are there per cohort?

In [37]:
if "COHORT" in df_status.columns:
    print("Cohorts (code):")
    print(df_status["COHORT"].value_counts(dropna=False))

if "COHORT_DEFINITION" in df_status.columns:
    print("\nCohorts (definition):")
    print(df_status["COHORT_DEFINITION"].value_counts(dropna=False))

if {"COHORT", "COHORT_DEFINITION"}.issubset(df_status.columns):
    print("\nUnique patients per cohort:")
    print(df_status.groupby("COHORT_DEFINITION")["PATNO"].nunique().sort_values(ascending=False))

Cohorts (code):
COHORT
4    5956
1    2142
2     440
3      81
Name: count, dtype: int64

Cohorts (definition):
COHORT_DEFINITION
Prodromal              5956
Parkinson's Disease    2142
Healthy Control         440
SWEDD                    81
Name: count, dtype: int64

Unique patients per cohort:
COHORT_DEFINITION
Prodromal              5956
Parkinson's Disease    2142
Healthy Control         440
SWEDD                    81
Name: PATNO, dtype: int64


### Duplicates in Participant_Status

In [38]:
dup_status = df_status[df_status.duplicated(subset=["PATNO"], keep=False)]
print("Rows with duplicate PATNO in Participant_Status:", len(dup_status))
if len(dup_status) > 0:
    display(dup_status.sort_values("PATNO").head(20))

Rows with duplicate PATNO in Participant_Status: 0


### What clinical columns does it contain and which are actually useful?

Columns with a low proportion of nulls (candidates for being useful) and those with a high proportion of nulls (probably not very useful or specific to a single visit) are automatically listed.

In [39]:
UMBRAL_UTIL = 40  # % of nulls below which it is considered "potentially useful"

columnas_utiles = resumen_status[resumen_status["pct_nulos"] < UMBRAL_UTIL].index.tolist()
columnas_dudosas = resumen_status[resumen_status["pct_nulos"] >= UMBRAL_UTIL].index.tolist()

print(f"Columns with <{UMBRAL_UTIL}% nulls (candidates for being useful): {len(columnas_utiles)}")
print(columnas_utiles)

print(f"\nColumns with >={UMBRAL_UTIL}% nulls (check if they add value): {len(columnas_dudosas)}")
print(columnas_dudosas)

Columns with <40% nulls (candidates for being useful): 30
['INEXPAGE', 'SCREENEDAM', 'ENRLOTHGV', 'SV2ASTDY', 'GAITSTDY', 'PISTDY', 'DPPDSTDY', 'NXTAUSTDY', 'DPPROSTDY', 'GAITLEAPSTDY', 'DPASTDY', 'DPDPASTDY', 'FD4STDY', 'TAUSTDY', 'ENRLPINK1', 'ENRLNORM', 'ENRLPRKN', 'ENRLSRDC', 'AV133STDY', 'ENROLL_STATUS', 'STATUS_DATE', 'PATNO', 'COHORT', 'COHORT_DEFINITION', 'PPMI_ONLINE_ENROLL', 'ENRLHPSM', 'ENRLRBD', 'ENRLLRRK2', 'ENRLSNCA', 'ENRLGBA']

Columns with >=40% nulls (check if they add value): 3
['DATELIG', 'ENROLL_AGE', 'ENROLL_DATE']


## 2. Loading Pacientes.csv (imaging information)

In [40]:
df_pacientes = cargar_csv(PACIENTES_CSV, "Pacientes.csv")
df_pacientes.head()


--- Pacientes.csv ---
Rows: 15326 | Columns: 4
Memory usage: 1.86 MB


,Subject ID,Sex,Age,Description
0,100001,M,67.4,SAG 3D MPRAGE
1,100001,M,67.4,MIDLINE SAG LOC
2,100001,M,67.4,RESTING STATE FMRI ep2d_fid_basic_bold
3,100001,M,67.4,RESTING STATE FMRI ep2d_fid_basic_bold
4,100001,M,67.4,AX T2 GRE MT


This block lists all columns present in the `df_pacientes` DataFrame, facilitating the identification of variables contained in this file.

In [41]:
print("Available columns in Pacientes.csv:\n")
for c in df_pacientes.columns:
    print(" -", c)

Available columns in Pacientes.csv:

 - Subject ID
 - Sex
 - Age
 - Description


In [42]:
CANDIDATOS_ID = ["PATNO", "Subject ID", "Subject", "SubjID", "SUBJECT_ID", "ID", "PatientID", "Patient Id"]

col_id_pacientes = None
for cand in CANDIDATOS_ID:
    if cand in df_pacientes.columns:
        col_id_pacientes = cand
        break

if col_id_pacientes is None:
    print("Identifier column not automatically detected.")
    print("Available columns:", list(df_pacientes.columns))
    print("=> Manually assign: col_id_pacientes = 'COLUMN_NAME'")
else:
    print("Identifier column detected in Pacientes.csv:", col_id_pacientes)

Identifier column detected in Pacientes.csv: Subject ID


In [43]:
# If automatic detection failed, uncomment and adjust this line:
# col_id_pacientes = "Subject ID"

def extraer_patno(valor):
    """Extracts only the digits from an identifier, in case it comes with prefixes (e.g., 'PPMI_3004' -> '3004')."""
    if pd.isna(valor):
        return np.nan
    match = re.search(r"(\d+)", str(valor))
    return int(match.group(1)) if match else np.nan

df_pacientes["PATNO_NORMALIZADO"] = df_pacientes[col_id_pacientes].apply(extraer_patno)
df_pacientes[[col_id_pacientes, "PATNO_NORMALIZADO"]].head()

,Subject ID,PATNO_NORMALIZADO
0,100001,100001
1,100001,100001
2,100001,100001
3,100001,100001
4,100001,100001


### Records and images per patient (Pacientes.csv brings repeated rows per study)

In [44]:
n_filas_pacientes = len(df_pacientes)
n_pacientes_unicos_img = df_pacientes["PATNO_NORMALIZADO"].nunique()

print("Total rows in Pacientes.csv (one row per study/acquisition):", n_filas_pacientes)
print("Unique patients with at least one imaging study:", n_pacientes_unicos_img)

if "Description" in df_pacientes.columns:
    print("\nMost common study types (Description column):")
    print(df_pacientes["Description"].value_counts().head(20))

estudios_por_paciente = df_pacientes.groupby("PATNO_NORMALIZADO").size().rename("n_estudios")
print("\nStudies per patient (statistical summary):")
print(estudios_por_paciente.describe())

Total rows in Pacientes.csv (one row per study/acquisition): 15326
Unique patients with at least one imaging study: 1525

Most common study types (Description column):
Description
2D GRE-MT             2456
Axial PD-T2 TSE FS     838
3D T2 FLAIR            743
MPRAGE GRAPPA          643
Axial PD-T2 TSE        638
AX T2 GRE MT           629
rsfMRI_RL              621
3D T1-weighted         578
2D GRE MT              430
AXIAL 2D GRE-MT        340
rsfMRI_PA              303
rsfMRI_AP              300
2D_GRE-MT              285
DTI_revB0_AP           226
2D GRE-NM              217
2D GRE-MT_ACPC         185
AX GRE -MT             175
2D GRE-NM_MT           160
T2                     154
NM-MT                  150
Name: count, dtype: int64

Studies per patient (statistical summary):
count    1525.000000
mean       10.049836
std        12.190047
min         1.000000
25%         4.000000
50%         7.000000
75%        13.000000
max       278.000000
Name: n_estudios, dtype: float64


## 3. Cross-referencing between Participant_Status and Pacientes

In [45]:
set_status = set(df_status["PATNO"].dropna().astype(int))
set_pacientes = set(df_pacientes["PATNO_NORMALIZADO"].dropna().astype(int))

en_ambos = set_status & set_pacientes
solo_status = set_status - set_pacientes
solo_pacientes = set_pacientes - set_status

print("Patients in Participant_Status:", len(set_status))
print("Patients in Pacientes.csv:", len(set_pacientes))
print("Patients in BOTH files:", len(en_ambos))
print("Patients ONLY in Participant_Status (no images):", len(solo_status))
print("Patients ONLY in Pacientes.csv (no status recorded):", len(solo_pacientes))

Patients in Participant_Status: 8619
Patients in Pacientes.csv: 1525
Patients in BOTH files: 1525
Patients ONLY in Participant_Status (no images): 7094
Patients ONLY in Pacientes.csv (no status recorded): 0


In [46]:
# Percentage of ACTIVE (Enrolled) patients who have imaging records
if "ENROLL_STATUS" in df_status.columns:
    set_activos = set(pacientes_activos["PATNO"].dropna().astype(int))
    activos_con_imagen = set_activos & set_pacientes
    pct_activos_con_imagen = 100 * len(activos_con_imagen) / len(set_activos) if len(set_activos) > 0 else 0

    print("Active patients (Enrolled):", len(set_activos))
    print("Active patients WITH images:", len(activos_con_imagen))
    print(f"Percentage of active patients with images: {pct_activos_con_imagen:.2f}%")

Active patients (Enrolled): 3999
Active patients WITH images: 1060
Percentage of active patients with images: 26.51%


In [47]:
# Relevant clinical columns that actually exist in the file (avoids errors if any are missing)
columnas_status_relevantes = [c for c in [
    "PATNO", "COHORT", "COHORT_DEFINITION", "ENROLL_STATUS", "ENROLL_DATE", "ENROLL_AGE"
] if c in df_status.columns]

base_status = df_status[columnas_status_relevantes].drop_duplicates(subset=["PATNO"]).copy()

resumen_imagenes = df_pacientes.groupby("PATNO_NORMALIZADO").agg(
    n_estudios_imagen=("PATNO_NORMALIZADO", "size")
).reset_index().rename(columns={"PATNO_NORMALIZADO": "PATNO"})

participant_index = base_status.merge(resumen_imagenes, on="PATNO", how="left")
participant_index["n_estudios_imagen"] = participant_index["n_estudios_imagen"].fillna(0).astype(int)
participant_index["tiene_imagenes"] = participant_index["n_estudios_imagen"] > 0

print("participant_index.csv -> rows:", len(participant_index))
participant_index.head(10)

participant_index.csv -> rows: 8619


,PATNO,COHORT,COHORT_DEFINITION,ENROLL_STATUS,ENROLL_DATE,ENROLL_AGE,n_estudios_imagen,tiene_imagenes
0,3000,2,Healthy Control,Withdrew,02/2011,69.1,2,True
1,3001,1,Parkinson's Disease,Withdrew,03/2011,65.1,3,True
2,3002,1,Parkinson's Disease,Withdrew,03/2011,67.6,3,True
3,3003,1,Parkinson's Disease,Enrolled,04/2011,56.7,3,True
4,3004,2,Healthy Control,Enrolled,04/2011,59.4,3,True
5,3005,1,Parkinson's Disease,Excluded,NaN,NaN,0,False
6,3006,1,Parkinson's Disease,Withdrew,05/2011,57.5,3,True
7,3007,1,Parkinson's Disease,Withdrew,05/2011,64.5,8,True
8,3008,2,Healthy Control,Withdrew,06/2011,81.9,3,True
9,3009,2,Healthy Control,Enrolled,06/2011,83.7,0,False


In [48]:
ruta_salida = os.path.join(RESULTADOS_DIR, "participant_index.csv")
participant_index.to_csv(ruta_salida, index=False)
print("Saved to:", ruta_salida)

Saved to: /content/drive/MyDrive/Investigación_Parkinson/Resultados/participant_index.csv


In [49]:
print("="*60)
print("SUMMARY — Notebook 1: Patient Analysis")
print("="*60)
print(f"Total patients (Participant_Status): {n_total_status}")
if 'set_activos' in dir():
    print(f"Enrolled patients (active): {len(set_activos)}")
print(f"Patients with imaging studies: {n_pacientes_unicos_img}")
print(f"Patients in both files: {len(en_ambos)}")
print(f"Patients only in Participant_Status: {len(solo_status)}")
print(f"Patients only in Pacientes.csv: {len(solo_pacientes)}")
print(f"Common identifier: PATNO")
print(f"Generated file: {ruta_salida}")
print("="*60)

SUMMARY — Notebook 1: Patient Analysis
Total patients (Participant_Status): 8619
Enrolled patients (active): 3999
Patients with imaging studies: 1525
Patients in both files: 1525
Patients only in Participant_Status: 7094
Patients only in Pacientes.csv: 0
Common identifier: PATNO
Generated file: /content/drive/MyDrive/Investigación_Parkinson/Resultados/participant_index.csv
